## Libraries and DataSet

In [15]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10 

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)

Using: cuda


In [7]:
# image -> scale (0,1) -> normalize  (-1,1) 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
]) 
# Created an transformer which will be applied to every single image

# Traning Dataset 
trainset = CIFAR10(
    root='./DATA',
    train=True,
    download=False,
    transform=transform
)

# testing Dataset 
testset = CIFAR10(
    root='./DATA',
    train=False,
    download=False,
    transform=transform
)

In [8]:
trainLoader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)
testLoader = DataLoader(
    testset,
    batch_size=64,
)

## Build the CNN ( Convolutional Neural Networks )


Input: (32,32,3)
CNN => ((Convo+ReLU) -> MaxPooling) => ((Convo+ReLU) -> MaxPooling) => Full Connected Layer 
Output => (10 layer because 10 Classes) + Linear Model 

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # Creating Layer 
        self.convo_layers =  nn.Sequential(
            # Layer 1
            nn.Conv2d(3, 32, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernal size=2, stride=2

            # Layer 2
            nn.Conv2d(32, 64, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernal size=2, stride=2
                
            # Layer 3
            nn.Conv2d(64, 128, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernal size=2, stride=2
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.convo_layers(x)
        x = x.view(x.size(0), -1) # Flattening 
        x = self.fc_layer(x)

        return x

In [10]:
model = CNN().to(device)

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Traning the CNN


In [16]:
epochs = 10

for epoch in range(epochs):
    training_loss = 0.0;

    for images, labels in trainLoader:
        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels)
        loss.backward() 

        optimizer.step()

        training_loss += loss.item()

    print(f"Epoch = {epoch+1} & Loss = {training_loss/len(trainLoader)}")
    

Epoch = 1 & Loss = 0.126837835736487
Epoch = 2 & Loss = 0.10022558865335096
Epoch = 3 & Loss = 0.09481086747666054
Epoch = 4 & Loss = 0.08707454786612116
Epoch = 5 & Loss = 0.08358434517808316
Epoch = 6 & Loss = 0.08003291080150839
Epoch = 7 & Loss = 0.07662104014986101
Epoch = 8 & Loss = 0.07092726176165168
Epoch = 9 & Loss = 0.06559229781225924
Epoch = 10 & Loss = 0.06906309583173145


In [21]:
# Evaluate CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testLoader:
        # Move test data to GPU
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = correct_labels / total_labels * 100

print(f"Accuracy Score: {accuracy:.2f}%")

Accuracy Score: 73.95%
